## Load Pre-Processed Data and Perform Splitting with Parameter Variations on SWSPy, for Comparison, then Save Results

In [ ]:
# Dependencies

# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import obspy
from obspy.core.utcdatetime import UTCDateTime
from obspy.clients.fdsn import Client
from obspy.core.event import read_events
import os
import sys
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Add local swspy directory to path (before other imports)
swspy_local_path = os.path.abspath('../swspy')
if swspy_local_path not in sys.path:
    sys.path.insert(0, swspy_local_path)

# Import swspy from local directory
import swspy

# Add scripts directory to path for custom modules
sys.path.append('.')
from get_all_traces import get_station_traces_batch
from splitting_functions import *
from teanby_clustering import *

# Set up plotting
plt.rcParams['figure.figsize'] = (12, 8)

print("Libraries imported successfully")
print(f"ObsPy version: {obspy.__version__}")
print(f"SWSPy available: {'Yes' if 'swspy' in sys.modules else 'No'}")
print(f"SWSPy location: {swspy.__file__}")

In [ ]:
# Reload the data: All AXAS2 evnts that passed QC across the week around the eruption onset
passing_waveforms = load_passing_waveforms(output_dir='passing_waveforms_data')

# Verify the reload worked correctly
print(f"Reloaded events: {len(passing_waveforms)}")
print(f"\nSample reloaded event (ID: {list(passing_waveforms.keys())[0]}):")
sample_event = passing_waveforms[list(passing_waveforms.keys())[0]]
print(f"  Station: {sample_event['station']}")
print(f"  Origin time: {sample_event['origin_time']}")
print(f"  Number of traces: {len(sample_event['traces'])}")
print(f"  Back azimuth: {sample_event['back_azimuth']:.2f}°")

In [ ]:
# Create a dataframe with quality control metrics for the test catalog events
qc_metrics_df = pd.DataFrame({
    'event_id': list(passing_waveforms.keys()),
    's_time': [passing_waveforms[eid]['s_arrival_time'] for eid in passing_waveforms.keys()],
    'station': [passing_waveforms[eid]['station'] for eid in passing_waveforms.keys()],
    'back_azimuth': [passing_waveforms[eid]['back_azimuth'] for eid in passing_waveforms.keys()],
    'snr_horizontal': [passing_waveforms[eid]['snr_horizontal'] for eid in passing_waveforms.keys()],
    'incidence': [passing_waveforms[eid]['incidence_eigenvalue_jurkevics'] for eid in passing_waveforms.keys()],
    'rectilinearity': [passing_waveforms[eid]['rectilinearity_jurkevics'] for eid in passing_waveforms.keys()],
    'latitude': [passing_waveforms[eid]['latitude'] for eid in passing_waveforms.keys()],
    'longitude': [passing_waveforms[eid]['longitude'] for eid in passing_waveforms.keys()],
    'depth': [passing_waveforms[eid]['depth'] for eid in passing_waveforms.keys()],
    'origin_time' : [passing_waveforms[eid]['origin_time'] for eid in passing_waveforms.keys()]
})

print(f"Quality control metrics dataframe created with {len(qc_metrics_df)} events")
display(qc_metrics_df)

In [ ]:
# Shear-wave splitting analysis using dynamic splitting parameters
print("Performing shear-wave splitting analysis using SWSPy method...")
results_swspy_sigma_3 = perform_splitting_on_organized_waveforms(passing_waveforms, first_window_start=3, last_window_start=0, 
                                                         first_window_end=1.5, last_window_end=2.5, n_win=10, s_pick_uncertainty=0.0509, mode='swspy', 
                                                         plot_results=False)


In [ ]:
def save_results_csv(results, file_name, file_loc='/Users/mhemmett/Seismology/axial-splitting-ml/results/'):
    # Create dataframe from results_swspy with event metadata and processing parameters

    # Initialize lists for each column
    data_rows = []

    for event_id, event_result in results.items():
        result = event_result['result']
        
        # Extract splitting measurements
        phi = result.get('phi')
        dt = result.get('dt')
        phi_error = result.get('phi_error')
        dt_error = result.get('dt_error')

        
        event_datetime = result.get('event_datetime')
        event_origin_time = result.get('event_origin_time')
        s_arrival_time = result.get('s_arrival_time')
        event_lat = result.get('event_lat')
        event_lon = result.get('event_lon')
        event_depth = result.get('event_depth')
        back_azimuth = result.get('back_azimuth')
        snr_horizontal = result.get('snr_horizontal')
        rectilinearity = result.get('rectilinearity_jurkevics')
        incidence = result.get('incidence_eigenvalue_jurkevics')
        first_window_start = result.get('first_window_start')
        last_window_start = result.get('last_window_start')
        first_window_end = result.get('first_window_end')
        last_window_end = result.get('last_window_end')
        n_win = result.get('n_win')
        s_pick_uncertainty = result.get('s_pick_uncertainty')
        dominant_period = result.get('dominant_period')

        # Calculate numerical values for window start and end times
        first_window_start_pre_S_pick =  first_window_start * s_pick_uncertainty
        last_window_start_pre_S_pick = last_window_start * s_pick_uncertainty
        first_window_end_post_S_pick =  first_window_end * dominant_period
        last_window_end_post_S_pick = last_window_end * dominant_period

        # Create row dictionary
        row = {
            'event_id': event_id,
            'event_datetime': event_datetime,
            's_arrival_time' : s_arrival_time,
            'event_origin_time' : event_origin_time,
            'event_lat': event_lat,
            'event_lon': event_lon,
            'event_depth': event_depth,
            'back_azimuth': back_azimuth,
            'incidence': incidence,
            'snr_horizontal': snr_horizontal,
            'rectilinearity': rectilinearity,
            'dominant_period': dominant_period,
            'phi': phi,
            'phi_error': phi_error,
            'dt': dt,
            'dt_error': dt_error,
            # Processing parameters (constant values as specified)
            'first_window_start_pre_S_pick': first_window_start_pre_S_pick,
            'last_window_start_pre_S_pick': last_window_start_pre_S_pick,
            'first_window_end_post_S_pick': first_window_end_post_S_pick,
            'last_window_end_post_S_pick': last_window_end_post_S_pick,
            'n_windows': n_win,
            'start_window_step_size': (first_window_start_pre_S_pick - last_window_start_pre_S_pick) / n_win,
            'end_window_step_size': (last_window_end_post_S_pick - first_window_end_post_S_pick) / n_win,
        }
        
        data_rows.append(row)

    # Create DataFrame
    results_df = pd.DataFrame(data_rows)

    # Display the dataframe
    print(f"\nSplitting Results DataFrame - {len(results_df)} events")
    print("=" * 100)
    display(results_df)

    # Save to CSV
    output_file = str(file_loc + file_name + '.csv')
    results_df.to_csv(output_file, index=False)
    print(f"\n✓ Saved to: {output_file}")

In [ ]:
save_results_csv(results_swspy_sigma_3, file_name='splitting_results_axas2_apr_20_28_3_sigma_start')

In [ ]:
# Plot rose diagram
if 'results_swspy_sigma_3' in locals() and results_swspy_sigma_3 is not None:
    # Create the rose plot
    fig, ax = plot_fast_direction_rose(
        results_swspy_sigma_3,
        title=f"Fast Direction Rose Plot - Station AXAS2, SWSPy: 3σ to S-pick Start, 1.5-2.5 Tmid End",
        nbins=18,  # 10° binsd
        color='steelblue',
        figsize=(10, 10)
    )
    plt.show()

# Plot timeseries of splitting parameters with smoothing
fig, axes, df = plot_splitting_timeseries_smooth(
    results_swspy_sigma_3,
    qc_metrics_df,
    station='AXAS2, SWSPy: 3σ to S-pick Start, 1.5-2.5 Tmid End',
    y_width_phi = 180 /20,  # 5 degrees converted to radians internally
    y_width_dt=2,   # 2 samples
    sigma=2.0
)
plt.show()

In [ ]:
# Shear-wave splitting analysis using dynamic splitting parameters
print("Performing shear-wave splitting analysis using SWSPy method...")
results_swspy_sigma_3_1 = perform_splitting_on_organized_waveforms(passing_waveforms, first_window_start=3, last_window_start=1, 
                                                         first_window_end=1.5, last_window_end=2.5, n_win=10, s_pick_uncertainty=0.0509, mode='swspy', 
                                                         plot_results=False)

save_results_csv(results_swspy_sigma_3_1, file_name='splitting_results_axas2_apr_20_28_3_sigma_start_1_sigma_end')

In [ ]:
# Plot rose diagram
if 'results_swspy_sigma_3_1' in locals() and results_swspy_sigma_3_1 is not None:
    # Create the rose plot
    fig, ax = plot_fast_direction_rose(
        results_swspy_sigma_3_1,
        title=f"Fast Direction Rose Plot - Station AXAS2, SWSPy: 3σ to 1σ Start, 1.5-2.5 Tmid End",
        nbins=18,  # 10° binsd
        color='steelblue',
        figsize=(10, 10)
    )
    plt.show()

# Plot timeseries of splitting parameters with smoothing
fig, axes, df = plot_splitting_timeseries_smooth(
    results_swspy_sigma_3_1,
    qc_metrics_df,
    station='AXAS2, SWSPy: 3σ to 1σ Start, 1.5-2.5 Tmid End',
    y_width_phi = 180 /20,  # 5 degrees converted to radians internally
    y_width_dt=2,   # 2 samples
    sigma=2.0
)
plt.show()

In [ ]:
# Shear-wave splitting analysis using dynamic splitting parameters
print("Performing shear-wave splitting analysis using SWSPy method...")
results_swspy_1_tmid_start = perform_splitting_on_organized_waveforms(passing_waveforms, first_window_start=1, last_window_start=0, 
                                                         first_window_end=1.5, last_window_end=2.5, n_win=10, s_pick_uncertainty=0.0509, mode='swspy', 
                                                         plot_results=False)

save_results_csv(results_swspy_1_tmid_start, file_name='splitting_results_axas2_apr_20_28_1_tmid_start')

In [ ]:
# Plot rose diagram
if 'results_swspy_1_tmid_start' in locals() and results_swspy_1_tmid_start is not None:
    # Create the rose plot
    fig, ax = plot_fast_direction_rose(
        results_swspy_1_tmid_start,
        title=f"Fast Direction Rose Plot - Station AXAS2, SWSPy: 1σ to S-pick Start, 1.0-2.5 Tmid End",
        nbins=18,  # 10° binsd
        color='steelblue',
        figsize=(10, 10)
    )
    plt.show()

# Plot timeseries of splitting parameters with smoothing
fig, axes, df = plot_splitting_timeseries_smooth(
    results_swspy_1_tmid_start,
    qc_metrics_df,
    station='AXAS2, SWSPy: 1σ to S-pick Start, 1.5-2.5 Tmid End',
    y_width_phi = 180 /20,  # 5 degrees converted to radians internally
    y_width_dt=2,   # 2 samples
    sigma=2.0
)
plt.show()

In [ ]:
# Shear-wave splitting analysis using dynamic splitting parameters
print("Performing shear-wave splitting analysis using SWSPy method...")
results_swspy_1_8_tmid_start_2_2_end = perform_splitting_on_organized_waveforms(passing_waveforms, first_window_start=2, last_window_start=0, 
                                                         first_window_end=1.8, last_window_end=2.2, n_win=10, s_pick_uncertainty=0.0509, mode='swspy', 
                                                         plot_results=False)

save_results_csv(results_swspy_1_8_tmid_start_2_2_end, file_name='splitting_results_axas2_apr_20_28_1_8_tmid_start_2_2_end')

In [ ]:
# Plot rose diagram
if 'results_swspy_1_5_tmid_start_3_end' in locals() and results_swspy_1_8_tmid_start_2_2_end is not None:
    # Create the rose plot
    fig, ax = plot_fast_direction_rose(
        results_swspy_1_8_tmid_start_2_2_end,
        title=f"Fast Direction Rose Plot - Station AXAS2, SWSPy: 2σ to S-pick Start, 1.8-2.2 Tmid End",
        nbins=18,  # 10° binsd
        color='steelblue',
        figsize=(10, 10)
    )
    plt.show()

# Plot timeseries of splitting parameters with smoothing
fig, axes, df = plot_splitting_timeseries_smooth(
    results_swspy_1_8_tmid_start_2_2_end,
    qc_metrics_df,
    station='AXAS2, SWSPy: 2σ to S-pick Start, 1.8-2.2 Tmid End',
    y_width_phi = 180 /20,  # 5 degrees converted to radians internally
    y_width_dt=2,   # 2 samples
    sigma=2.0
)
plt.show()

In [ ]:
save_results_csv(results_swspy_1_8_tmid_start_2_2_end, file_name='splitting_results_axas2_apr_20_28_1_8_tmid_start_2_2_end')

In [ ]:
# Shear-wave splitting analysis using dynamic splitting parameters
print("Performing shear-wave splitting analysis using SWSPy method...")
results_swspy_1_5_tmid_start_3_5_end = perform_splitting_on_organized_waveforms(passing_waveforms, first_window_start=2, last_window_start=0, 
                                                         first_window_end=1.5, last_window_end=2.6, n_win=10, s_pick_uncertainty=0.0509, mode='swspy', 
                                                         plot_results=False)

save_results_csv(results_swspy_1_5_tmid_start_3_5_end, file_name='splitting_results_axas2_apr_20_28_1.5_tmid_start_3_end')

In [ ]:
# Plot rose diagram
if 'results_swspy_1.5_tmid_start_3_end' in locals() and results_swspy_1.5_tmid_start_3_end is not None:
    # Create the rose plot
    fig, ax = plot_fast_direction_rose(
        results_swspy_1.5_tmid_start_3_end,
        title=f"Fast Direction Rose Plot - Station AXAS2, SWSPy: 2σ to S-pick Start, 1.5-3.0 Tmid End",
        nbins=18,  # 10° binsd
        color='steelblue',
        figsize=(10, 10)
    )
    plt.show()

# Plot timeseries of splitting parameters with smoothing
fig, axes, df = plot_splitting_timeseries_smooth(
    results_swspy_1.5_tmid_start_3_end,
    qc_metrics_df,
    station='AXAS2, SWSPy: 2σ to S-pick Start, 1.5-3.0 Tmid End',
    y_width_phi = 180 /20,  # 5 degrees converted to radians internally
    y_width_dt=2,   # 2 samples
    sigma=2.0
)
plt.show()